In [16]:
from nuscenes.nuscenes import NuScenes
from nuscenes.scripts.export_poses import derive_latlon

from bst_id.encoder import BSTIDEncoder
from bst_id.decoder import BSTIDDecoder


dataroot = "../data/nuscenes"

nusc = NuScenes(
    version="v1.0-mini",
    dataroot=dataroot,
    verbose=True
)

print("Scenes:", len(nusc.scene))
print(nusc.scene[0])

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.205 seconds.
Reverse indexing ...
Done reverse indexing in 0.0 seconds.
Scenes: 10
{'token': 'cc8c0bf57f984915a77078b10eb33198', 'log_token': '7e25a2c8ea1f41c5b0da1e69ecfa71a2', 'nbr_samples': 39, 'first_sample_token': 'ca9a282c9e77460f8360f564131a8af5', 'last_sample_token': 'ed5fc18c31904f96a8f0dbb99ff069c0', 'name': 'scene-0061', 'description': 'Parked truck, construction, intersection, turn left, following a van'}


### 1. Pick a scene

In [17]:
scene = nusc.scene[0]

sample = nusc.get(
    "sample",
    scene["first_sample_token"]
)

print(sample.keys())
print(sample["data"].keys())

dict_keys(['token', 'timestamp', 'prev', 'next', 'scene_token', 'data', 'anns'])
dict_keys(['RADAR_FRONT', 'RADAR_FRONT_LEFT', 'RADAR_FRONT_RIGHT', 'RADAR_BACK_LEFT', 'RADAR_BACK_RIGHT', 'LIDAR_TOP', 'CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_BACK_RIGHT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_FRONT_LEFT'])


### 2. Determine which global HD map this scene uses

In [18]:
log = nusc.get("log", scene["log_token"])

location = log["location"]

print("Map:", location)


Map: singapore-onenorth


### 3. Get ego pose

In [19]:
cam_token = sample["data"]["CAM_FRONT"]

sample_data = nusc.get(
    "sample_data",
    cam_token
)

ego_pose = nusc.get(
    "ego_pose",
    sample_data["ego_pose_token"]
)

x, y, z = ego_pose["translation"]

print("Global position:")
print("x =", x)
print("y =", y)
print("z =", z)

Global position:
x = 411.4199861830012
y = 1181.197175631848
z = 0.0


### 4. Convert nuScenes global XY → latitude/longitude

In [20]:
latlon = derive_latlon(
    location=location,
    poses=[ego_pose]
)[0]

latitude = latlon["latitude"]
longitude = latlon["longitude"]

print()
print("Geographic position:")
print("longitude =", longitude)
print("latitude  =", latitude)


Geographic position:
longitude = 103.78844869048285
latitude  = 1.298820958950809


### 6. Encode with actual BST-ID library

In [21]:
ZOOM = 25

bst_id, bit_len = BSTIDEncoder.encode(
    x=longitude,
    y=latitude,
    f=None,
    t_unix=None,
    zoom_x=ZOOM,
    zoom_y=ZOOM,
)

print()
print("BST-ID:")
print(bin(bst_id))

print("Bit length:")
print(bit_len)



BST-ID:
0b1100110001100011001001110011100001110000111111100010011100010010
Bit length:
64


### 7. Decode

In [22]:
decoded_lon, decoded_lat, decoded_f, decoded_t = \
    BSTIDDecoder.decode(
        bst_id,
        bit_len
    )

print()
print("Decoded BST cell:")
print("longitude =", decoded_lon)
print("latitude  =", decoded_lat)


Decoded BST cell:
longitude = 103.78844261169434
latitude  = 1.2988289278585083
